In [1]:
!pip -q install ollama neo4j psycopg2-binary sqlparse tqdm

In [2]:
import os, json, re, time, hashlib
import pandas as pd
import psycopg2
import sqlparse
import ollama
from neo4j import GraphDatabase
from tqdm import tqdm

import datetime
import decimal
import uuid

In [3]:
# -----------------------------
# Files
# -----------------------------
CSV_PATH = "final_test_data.csv"
OUT_REPORT = "nl2sql_test_report_with_results_EnrichKG.csv"
OUT_SUMMARY = "nl2sql_test_report_summary_EnrichKG.csv"

# -----------------------------
# Postgres
# -----------------------------
PG_HOST = os.getenv("PG_HOST", "localhost")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB   = os.getenv("PG_DB", "customer_orders_and_reviews_db")
PG_USER = os.getenv("PG_USER", "postgres")
PG_PASS = os.getenv("PG_PASS", "postgres")

# -----------------------------
# Neo4j
# -----------------------------
NEO4J_URI  = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASS = os.getenv("NEO4J_PASS", "neo4jpassword")

# -----------------------------
# Ollama
# -----------------------------
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma3:4b")   # e.g. gemma2:9b
TEMP_TABLES  = 0.0
TEMP_COLS    = 0.0
TEMP_SQL     = 0.1

# -----------------------------
# Evaluation
# -----------------------------
FETCH_ROWS = 20


In [4]:
pg_conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASS
)
pg_conn.autocommit = True
print("Postgres connected ✅")

neo_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
print("Neo4j connected ✅")


Postgres connected ✅
Neo4j connected ✅


In [5]:
df_tests = pd.read_csv(CSV_PATH)

def parse_params(x):
    if pd.isna(x) or str(x).strip() == "":
        return {}
    try:
        return json.loads(x)
    except Exception:
        return {}

print("Loaded tests:", len(df_tests))
df_tests.head(3)


Loaded tests: 75


,id,complexity,question,expected_sql
0,1,easy,List all products.,SELECT * FROM products;
1,2,easy,List all products in the 'Electronics' category.,SELECT * FROM public.products WHERE category =...
2,3,easy,Find all products priced above 100 dollars.,SELECT * FROM public.products WHERE price > 100;


In [6]:
FULL_SCHEMA_TEXT = """
TABLE public.customers
  - created_at (timestamp without time zone)
  - zip_code (character varying)
  - country (character varying)
  - state (character varying)
  - city (character varying)
  - address (text)
  - phone (character varying)
  - email (character varying) NOT_NULL
  - last_name (character varying) NOT_NULL
  - first_name (character varying) NOT_NULL
  - customer_id (uuid) PK NOT_NULL

TABLE public.order_items
  - created_at (timestamp without time zone)
  - subtotal (numeric) NOT_NULL
  - unit_price (numeric) NOT_NULL
  - quantity (integer) NOT_NULL
  - product_id (uuid) NOT_NULL
  - order_id (uuid) NOT_NULL
  - order_item_id (uuid) PK NOT_NULL

TABLE public.orders
  - notes (text)
  - shipping_address (text)
  - total_amount (numeric) NOT_NULL
  - status (character varying) NOT_NULL
  - order_date (timestamp without time zone)
  - customer_id (uuid) NOT_NULL
  - order_id (uuid) PK NOT_NULL

TABLE public.products
  - updated_at (timestamp without time zone)
  - created_at (timestamp without time zone)
  - stock_quantity (integer) NOT_NULL
  - price (numeric) NOT_NULL
  - category (character varying) NOT_NULL
  - description (text)
  - product_name (character varying) NOT_NULL
  - product_id (uuid) PK NOT_NULL

TABLE public.reviews
  - created_at (timestamp without time zone)
  - review_text (text)
  - rating (integer) NOT_NULL
  - customer_id (uuid) NOT_NULL
  - product_id (uuid) NOT_NULL
  - review_id (uuid) PK NOT_NULL

FOREIGN_KEYS
  - public.order_items.order_id -> public.orders.order_id
  - public.order_items.product_id -> public.products.product_id
  - public.orders.customer_id -> public.customers.customer_id
  - public.reviews.product_id -> public.products.product_id
  - public.reviews.customer_id -> public.customers.customer_id
""".strip()

In [7]:
def neo_run(cypher, params=None):
    with neo_driver.session() as session:
        return session.run(cypher, params or {}).data()


In [8]:
KG_ENRICH_QUERIES = [
"""
MATCH (t:Table {full_name:'public.customers'})
SET t.description = 'Customer master data: identity, contact, and location fields'
""",
"""
MATCH (t:Table {full_name:'public.orders'})
SET t.description = 'Orders placed by customers with status, date, shipping address and total amount'
""",
"""
MATCH (t:Table {full_name:'public.order_items'})
SET t.description = 'Line items for orders; each line references a product, with quantity and pricing'
""",
"""
MATCH (t:Table {full_name:'public.products'})
SET t.description = 'Product catalog: name, category, price and inventory'
""",
"""
MATCH (t:Table {full_name:'public.reviews'})
SET t.description = 'Customer reviews for products with rating and optional review text'
""",
# columns
"""
MATCH (t:Table {full_name:'public.orders'})-[:HAS_COLUMN]->(c:Column {name:'status'})
SET c.description='Order lifecycle status (pending, processing, shipped, delivered)',
    c.synonyms=['state','order state','delivery status'],
    c.examples=['pending','processing','shipped','delivered']
""",
"""
MATCH (t:Table {full_name:'public.orders'})-[:HAS_COLUMN]->(c:Column {name:'total_amount'})
SET c.description='Total amount for the order (numeric), often sum of order_items.subtotal',
    c.synonyms=['revenue','total price','order amount','amount paid']
""",
"""
MATCH (t:Table {full_name:'public.order_items'})-[:HAS_COLUMN]->(c:Column {name:'subtotal'})
SET c.description='Line subtotal = quantity * unit_price',
    c.synonyms=['line total','item total']
""",
"""
MATCH (t:Table {full_name:'public.reviews'})-[:HAS_COLUMN]->(c:Column {name:'rating'})
SET c.description='Integer rating score (typically 1-5)',
    c.synonyms=['stars','score','review rating'],
    c.examples=[1,2,3,4,5]
""",
"""
MATCH (t:Table {full_name:'public.products'})-[:HAS_COLUMN]->(c:Column {name:'stock_quantity'})
SET c.description='Inventory units available',
    c.synonyms=['stock','inventory','qty in stock','availability']
""",
]

for q in KG_ENRICH_QUERIES:
    neo_run(q)

print("KG enrichment applied ✅")

KG enrichment applied ✅


In [9]:
def ollama_generate(prompt: str, temperature=0.0, model=OLLAMA_MODEL) -> str:
    resp = ollama.generate(
        model=model,
        prompt=prompt,
        options={"temperature": float(temperature)},
        stream=False
    )
    return (resp.get("response") or "").strip()

def extract_json(text: str):
    # direct parse
    try:
        return json.loads(text)
    except Exception:
        # find the first {...}
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if not m:
            raise ValueError("No JSON found in LLM output")
        return json.loads(m.group(0))

In [10]:
def normalize_sql(sql: str) -> str:
    sql = (sql or "").strip()
    if not sql:
        return ""
    sql = sql.strip().strip("`")
    sql = sql.split(";")[0].strip() + ";"
    sql = sqlparse.format(sql, keyword_case="upper", strip_comments=True, reindent=False)
    sql = re.sub(r"\s+", " ", sql).strip()
    return sql

def json_safe(obj):
    if isinstance(obj, (datetime.datetime, datetime.date)):
        return obj.isoformat()
    if isinstance(obj, decimal.Decimal):
        return float(obj)
    if isinstance(obj, uuid.UUID):
        return str(obj)
    if isinstance(obj, (bytes, bytearray)):
        return obj.decode("utf-8", errors="replace")
    return str(obj)

def dumps_safe(obj) -> str:
    return json.dumps(obj, ensure_ascii=False, default=json_safe)

def rows_to_jsonable(cols, rows):
    if cols is None or rows is None:
        return None
    return [{cols[i]: json_safe(row[i]) for i in range(len(cols))} for row in rows]

def rows_fingerprint(cols, rows):
    if rows is None:
        return None
    m = hashlib.sha256()
    m.update(repr(cols).encode("utf-8"))
    m.update(repr(rows).encode("utf-8"))
    return m.hexdigest()

In [11]:
def try_execute(sql: str, fetch_rows: int = FETCH_ROWS):
    cur = None
    try:
        cur = pg_conn.cursor()
        cur.execute(sql)
        cols, rows = None, None
        if cur.description is not None:
            cols = [d[0] for d in cur.description]
            rows = cur.fetchmany(fetch_rows)
        cur.close()
        return True, cols, rows, None
    except Exception as e:
        try:
            if cur:
                cur.close()
        except Exception:
            pass
        return False, None, None, str(e)


In [12]:
def result_diff_hint(exp_ok, exp_cols, exp_rows, exp_err,
                     gen_ok, gen_cols, gen_rows, gen_err,
                     exact_match, result_match):
    if gen_err and str(gen_err).strip().lower() == "empty sql":
        return "empty sql"
    if not gen_ok:
        return f"generated sql execution failed: {gen_err}"
    if not exp_ok:
        return f"expected sql execution failed: {exp_err}"

    if exp_cols is None and gen_cols is None:
        return "both queries returned no result set"

    if (exp_cols is None) != (gen_cols is None):
        return "one query returned rows, the other did not"

    if exp_cols != gen_cols:
        return "column mismatch"

    exp_cnt = 0 if exp_rows is None else len(exp_rows)
    gen_cnt = 0 if gen_rows is None else len(gen_rows)

    if exp_cnt != gen_cnt:
        return "row count mismatch (sampled)"

    if result_match:
        return "ok"
    if exact_match:
        return "sql matches but sample differs"

    return "sample rows mismatch"


In [13]:
TABLE_SELECTOR_PROMPT = """You are a PostgreSQL schema expert.

Return STRICT JSON only:
{
  "tables": ["public.table1","public.table2",...],
  "reason": {"public.table1":"...", "public.table2":"..."}
}

Rules:
- Choose ONLY tables required to answer the question.
- If mentions rating/reviews -> reviews/products.
- If mentions order total/revenue/status/date -> orders (and customers for customer fields).
- If mentions quantity/subtotal/unit_price -> order_items (+ orders/products as needed).
- If mentions customer info -> customers.
- Do NOT invent tables.
- JSON only. No markdown.
"""

def llm_select_tables(question: str) -> dict:
    prompt = f"""{TABLE_SELECTOR_PROMPT}

SCHEMA:
{FULL_SCHEMA_TEXT}

QUESTION:
{question}

JSON:
"""
    raw = ollama_generate(prompt, temperature=TEMP_TABLES)
    return extract_json(raw)

In [14]:
def kg_fetch_tables_with_metadata(tables: list):
    cols = neo_run("""
    MATCH (t:Table)-[:HAS_COLUMN]->(c:Column)
    WHERE t.full_name IN $tables
    RETURN t.full_name AS table,
           c.name AS col,
           coalesce(c.data_type,'') AS data_type,
           coalesce(c.nullable,true) AS nullable,
           coalesce(c.is_pk,false) AS is_pk,
           coalesce(c.description,'') AS description,
           coalesce(c.synonyms,[]) AS synonyms,
           coalesce(c.examples,[]) AS examples
    ORDER BY table, col
    """, {"tables": tables})

    joins = neo_run("""
    MATCH (c1:Column)-[:FK_TO]->(c2:Column)
    MATCH (t1:Table)-[:HAS_COLUMN]->(c1)
    MATCH (t2:Table)-[:HAS_COLUMN]->(c2)
    WHERE t1.full_name IN $tables AND t2.full_name IN $tables
    RETURN t1.full_name AS from_table, c1.name AS from_col,
           t2.full_name AS to_table, c2.name AS to_col
    """, {"tables": tables})

    # dedupe joins
    seen = set()
    uniq = []
    for j in joins:
        k = (j["from_table"], j["from_col"], j["to_table"], j["to_col"])
        if k not in seen:
            seen.add(k)
            uniq.append(j)

    return cols, uniq

def render_table_schemas(cols_rows, joins_rows):
    by_table = {}
    for r in cols_rows:
        by_table.setdefault(r["table"], []).append(r)

    lines = []
    for t, cols in by_table.items():
        lines.append(f"TABLE {t}")
        for c in cols:
            pk = " PK" if c["is_pk"] else ""
            nn = " NOT_NULL" if (c["nullable"] is False) else ""
            desc = f" -- {c['description']}" if c["description"] else ""
            lines.append(f"  - {c['col']} ({c['data_type']}){pk}{nn}{desc}")
            if c.get("synonyms"):
                lines.append(f"    synonyms: {', '.join(c['synonyms'])}")
            if c.get("examples"):
                lines.append(f"    examples: {c['examples']}")

    if joins_rows:
        lines.append("\nALLOWED_JOINS (use only these):")
        for j in joins_rows:
            lines.append(f"  - {j['from_table']}.{j['from_col']} = {j['to_table']}.{j['to_col']}")

    return "\n".join(lines)


In [15]:
COLUMN_SELECTOR_PROMPT = """You select the minimal columns needed to write correct SQL.

Return STRICT JSON only:
{
  "columns": {
     "public.table1": ["colA","colB",...],
     ...
  },
  "needs_aggregation": true/false,
  "group_by": ["public.t.col",...],
  "order_by": [{"column":"public.t.col","direction":"ASC|DESC"}],
  "limit": null|number,
  "filters": [{"column":"public.t.col","op":"=","value_hint":"..."}]
}

Rules:
- Use only columns present in TABLE_SCHEMAS below.
- Choose only columns needed for SELECT/JOIN/WHERE/GROUP BY/ORDER BY.
- SELECT * rule:
  If question contains "list all" / "show all" / "find all" / "get all"
  and user did NOT ask for specific fields, set the main table columns to ["*"].
- If aggregation is requested, do NOT use "*".
- JSON only.
"""

def llm_select_columns(question: str, table_schema_text: str) -> dict:
    prompt = f"""{COLUMN_SELECTOR_PROMPT}

TABLE_SCHEMAS:
{table_schema_text}

QUESTION:
{question}

JSON:
"""
    raw = ollama_generate(prompt, temperature=TEMP_COLS)
    return extract_json(raw)


In [16]:
def kg_fetch_minimal_schema(selected_tables, selected_columns_map, joins_rows):
    # join columns required
    join_cols = {}
    for j in joins_rows:
        join_cols.setdefault(j["from_table"], set()).add(j["from_col"])
        join_cols.setdefault(j["to_table"], set()).add(j["to_col"])

    # get all columns for selected tables
    cols_rows = neo_run("""
    MATCH (t:Table)-[:HAS_COLUMN]->(c:Column)
    WHERE t.full_name IN $tables
    RETURN t.full_name AS table,
           c.name AS col,
           coalesce(c.data_type,'') AS data_type,
           coalesce(c.nullable,true) AS nullable,
           coalesce(c.is_pk,false) AS is_pk,
           coalesce(c.description,'') AS description
    """, {"tables": selected_tables})

    by_table = {}
    for r in cols_rows:
        by_table.setdefault(r["table"], []).append(r)

    filtered = {}
    for t in selected_tables:
        requested = set(selected_columns_map.get(t, []))
        star = "*" in requested
        if star:
            requested = set()

        # always include PK + join cols
        for c in by_table.get(t, []):
            if c["is_pk"]:
                requested.add(c["col"])
        for jc in join_cols.get(t, set()):
            requested.add(jc)

        if star:
            filtered[t] = {"star": True, "cols": [c for c in by_table.get(t, []) if c["col"] in requested]}
        else:
            filtered[t] = {"star": False, "cols": [c for c in by_table.get(t, []) if c["col"] in requested]}

    return filtered

def render_minimal_schema(filtered, joins_rows):
    lines = []
    for t, info in filtered.items():
        lines.append(f"TABLE {t}")
        if info["star"]:
            lines.append("  - * (all columns)")
        for c in info["cols"]:
            pk = " PK" if c["is_pk"] else ""
            nn = " NOT_NULL" if (c["nullable"] is False) else ""
            desc = f" -- {c['description']}" if c["description"] else ""
            lines.append(f"  - {c['col']} ({c['data_type']}){pk}{nn}{desc}")

    if joins_rows:
        lines.append("\nALLOWED_JOINS (use only these):")
        for j in joins_rows:
            lines.append(f"  - {j['from_table']}.{j['from_col']} = {j['to_table']}.{j['to_col']}")

    return "\n".join(lines)


In [17]:
SQL_GEN_PROMPT = """You are a senior PostgreSQL data engineer.

Generate exactly ONE valid PostgreSQL SQL statement.

STRICT RULES:
1) Return ONLY SQL. No explanation/markdown/comments.
2) Use schema-qualified names exactly as provided (e.g., public.orders).
3) Use ONLY tables/columns listed in SCHEMA below.
4) Use ONLY joins listed under ALLOWED_JOINS.
5) SELECT * rule:
   - If question contains "list all" / "show all" / "find all" / "get all"
     and user did NOT ask specific fields,
     then use SELECT * for the main table(s).
   - Otherwise select only required columns.
6) If aggregation is requested, do NOT use SELECT *.
7) End with semicolon.
"""

def llm_generate_sql(question: str, minimal_schema_text: str) -> str:
    prompt = f"""{SQL_GEN_PROMPT}

SCHEMA:
{minimal_schema_text}

QUESTION:
{question}

SQL:
"""
    raw = ollama_generate(prompt, temperature=TEMP_SQL)
    return normalize_sql(raw)


In [18]:
def nl2sql_pipeline(question: str):
    # Pass 1: tables
    tsel = llm_select_tables(question)
    tables = tsel.get("tables", [])
    if not tables:
        # fallback to all
        tables = ["public.customers","public.orders","public.order_items","public.products","public.reviews"]

    # KG fetch: all cols + joins
    cols_rows, joins_rows = kg_fetch_tables_with_metadata(tables)
    table_schema_text = render_table_schemas(cols_rows, joins_rows)

    # Pass 2: required columns
    csel = llm_select_columns(question, table_schema_text)
    columns_map = csel.get("columns", {}) or {}

    # KG minimal schema
    filtered = kg_fetch_minimal_schema(tables, columns_map, joins_rows)
    minimal_schema_text = render_minimal_schema(filtered, joins_rows)

    # Pass 3: SQL
    sql = llm_generate_sql(question, minimal_schema_text)

    return {
        "tables_selected": tables,
        "table_selector_json": tsel,
        "column_selector_json": csel,
        "minimal_schema_text": minimal_schema_text,
        "sql": sql
    }


In [19]:
out = nl2sql_pipeline("List all delivered orders")
print("Tables:", out["tables_selected"])
print("\nSQL:\n", out["sql"])

Tables: ['public.orders']

SQL:
 SELECT * FROM public.orders WHERE status = 'delivered';


In [20]:
results = []

for _, r in tqdm(df_tests.iterrows(), total=len(df_tests)):
    qid = int(r["id"])
    complexity = str(r["complexity"])
    question = str(r["question"])
    expected_sql = normalize_sql(str(r["expected_sql"]).strip())

    # Generate SQL using your KG pipeline
    t0 = time.time()
    try:
        pipe = nl2sql_pipeline(question)
        generated_sql = pipe["sql"]
        llm_err = ""
        tables_selected = pipe["tables_selected"]
        table_sel_json = dumps_safe(pipe["table_selector_json"])
        col_sel_json = dumps_safe(pipe["column_selector_json"])
        minimal_schema_text = pipe["minimal_schema_text"]
    except Exception as e:
        generated_sql = ""
        llm_err = str(e)
        tables_selected = []
        table_sel_json = ""
        col_sel_json = ""
        minimal_schema_text = ""

    llm_latency_ms = int((time.time() - t0) * 1000)

    # Execute expected + generated
    exp_ok, exp_cols, exp_rows, exp_err = try_execute(expected_sql, fetch_rows=FETCH_ROWS)

    if generated_sql:
        gen_ok, gen_cols, gen_rows, gen_err = try_execute(generated_sql, fetch_rows=FETCH_ROWS)
    else:
        gen_ok, gen_cols, gen_rows, gen_err = False, None, None, "Empty SQL"

    expected_row_count = len(exp_rows) if exp_rows is not None else (0 if exp_ok else None)
    generated_row_count = len(gen_rows) if gen_rows is not None else (0 if gen_ok else None)

    exp_sample = rows_to_jsonable(exp_cols, exp_rows)
    gen_sample = rows_to_jsonable(gen_cols, gen_rows)

    exp_sample_json = dumps_safe(exp_sample) if exp_sample is not None else ""
    gen_sample_json = dumps_safe(gen_sample) if gen_sample is not None else ""

    exact_match = (expected_sql == generated_sql) if generated_sql else False

    result_match = False
    if exp_ok and gen_ok:
        result_match = rows_fingerprint(exp_cols, exp_rows) == rows_fingerprint(gen_cols, gen_rows)

    hint = result_diff_hint(
        exp_ok, exp_cols, exp_rows, exp_err,
        gen_ok, gen_cols, gen_rows, gen_err,
        exact_match, result_match
    )

    generated_ok = bool(gen_ok and (result_match or exact_match))

    results.append({
        "id": qid,
        "complexity": complexity,
        "question": question,

        "expected_sql": expected_sql,
        "generated_sql": generated_sql,

        "llm_latency_ms": llm_latency_ms,
        "llm_error": llm_err,

        "tables_selected": dumps_safe(tables_selected),
        "table_selector_json": table_sel_json,
        "column_selector_json": col_sel_json,
        "minimal_schema_text": minimal_schema_text,

        "expected_exec_ok": exp_ok,
        "expected_exec_error": exp_err or "",
        "generated_exec_ok": gen_ok,
        "generated_exec_error": gen_err or "",

        "expected_row_count_sample": expected_row_count,
        "generated_row_count_sample": generated_row_count,

        "expected_result_sample_json": exp_sample_json,
        "generated_result_sample_json": gen_sample_json,

        "exact_match": exact_match,
        "result_match_sample": result_match,
        "result_diff_hint": hint,

        "generated_ok": generated_ok
    })

df_report = pd.DataFrame(results)
df_report.head(3)


100%|███████████████████████████████████████████| 75/75 [15:24<00:00, 12.33s/it]


,id,complexity,question,expected_sql,generated_sql,llm_latency_ms,llm_error,tables_selected,table_selector_json,column_selector_json,...,generated_exec_ok,generated_exec_error,expected_row_count_sample,generated_row_count_sample,expected_result_sample_json,generated_result_sample_json,exact_match,result_match_sample,result_diff_hint,generated_ok
0,1,easy,List all products.,SELECT * FROM products;,SELECT * FROM public.products;,7645,,"[""public.products""]","{""tables"": [""public.products""], ""reason"": {""pu...","{""columns"": {""public.products"": [""product_id"",...",...,True,,20.0,20.0,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...",False,True,ok,True
1,2,easy,List all products in the 'Electronics' category.,SELECT * FROM public.products WHERE category =...,SELECT * FROM public.products WHERE category =...,8724,,"[""public.products""]","{""tables"": [""public.products""], ""reason"": {""pu...","{""columns"": {""public.products"": [""product_id"",...",...,True,,9.0,9.0,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...",True,True,ok,True
2,3,easy,Find all products priced above 100 dollars.,SELECT * FROM public.products WHERE price > 100;,SELECT * FROM public.products WHERE price > 100;,8558,,"[""public.products""]","{""tables"": [""public.products""], ""reason"": {""pu...","{""columns"": {""public.products"": [""product_id"",...",...,True,,8.0,8.0,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...",True,True,ok,True


In [21]:
df_report.to_csv(OUT_REPORT, index=False)
print("Saved detailed report ✅:", OUT_REPORT)

Saved detailed report ✅: nl2sql_test_report_with_results_EnrichKG.csv


In [22]:
summary = df_report.groupby("complexity").agg(
    total=("id", "count"),
    exact_acc=("exact_match","mean"),
    exec_ok_rate=("generated_exec_ok","mean"),
    result_acc=("result_match_sample","mean"),
    pass_rate=("generated_ok","mean"),
    avg_llm_ms=("llm_latency_ms","mean"),
).reset_index()

order = ["easy", "medium", "hard"]
summary["complexity"] = pd.Categorical(summary["complexity"], categories=order, ordered=True)
summary = summary.sort_values("complexity").reset_index(drop=True)

summary.to_csv(OUT_SUMMARY, index=False)
print("Saved summary ✅:", OUT_SUMMARY)

summary

Saved summary ✅: nl2sql_test_report_summary_EnrichKG.csv


,complexity,total,exact_acc,exec_ok_rate,result_acc,pass_rate,avg_llm_ms
0,easy,25,0.16,1.00,0.80,0.80,10007.36
1,medium,25,0.00,0.80,0.28,0.28,12273.00
2,hard,25,0.00,0.72,0.16,0.16,14666.28


In [23]:
def generate_sql_with_retry(question: str):
    """
    Returns:
      sql (str),
      meta (dict) including minimal_schema_text and retry details
    """
    pipe = nl2sql_pipeline(question)
    sql1 = pipe["sql"]
    minimal_schema_text = pipe["minimal_schema_text"]

    # Try execute SQL1
    ok1, cols1, rows1, err1 = try_execute(sql1, fetch_rows=FETCH_ROWS) if sql1 else (False, None, None, "Empty SQL")

    # If ok, done
    if ok1:
        pipe["retry_used"] = False
        pipe["final_sql"] = sql1
        pipe["final_exec_ok"] = True
        pipe["final_exec_error"] = ""
        return sql1, pipe

    # Retry once using DB error feedback
    sql2 = llm_repair_sql(
        question=question,
        minimal_schema_text=minimal_schema_text,
        failed_sql=sql1,
        db_error=err1 or ""
    )

    ok2, cols2, rows2, err2 = try_execute(sql2, fetch_rows=FETCH_ROWS) if sql2 else (False, None, None, "Empty SQL")

    pipe["retry_used"] = True
    pipe["sql_before_retry"] = sql1
    pipe["sql_after_retry"] = sql2
    pipe["retry_error_before"] = err1 or ""
    pipe["retry_error_after"] = err2 or ""
    pipe["final_sql"] = sql2
    pipe["final_exec_ok"] = ok2
    pipe["final_exec_error"] = err2 or ""

    return sql2, pipe


In [24]:
results = []

for _, r in tqdm(df_tests.iterrows(), total=len(df_tests)):
    qid = int(r["id"])
    complexity = str(r["complexity"])
    question = str(r["question"])
    expected_sql = normalize_sql(str(r["expected_sql"]).strip())

    # Generate SQL using KG pipeline + retry
    t0 = time.time()
    try:
        generated_sql, pipe = generate_sql_with_retry(question)
        llm_err = ""

        tables_selected = pipe.get("tables_selected", [])
        table_sel_json = dumps_safe(pipe.get("table_selector_json", {})) if isinstance(pipe.get("table_selector_json"), dict) else dumps_safe(pipe.get("table_selector_json", {}))
        col_sel_json = dumps_safe(pipe.get("column_selector_json", {})) if isinstance(pipe.get("column_selector_json"), dict) else dumps_safe(pipe.get("column_selector_json", {}))

        minimal_schema_text = pipe.get("minimal_schema_text", "")

        retry_used = pipe.get("retry_used", False)
        sql_before_retry = pipe.get("sql_before_retry", "")
        sql_after_retry = pipe.get("sql_after_retry", "")
        retry_error_before = pipe.get("retry_error_before", "")
        retry_error_after = pipe.get("retry_error_after", "")

    except Exception as e:
        generated_sql = ""
        pipe = {}
        llm_err = str(e)

        tables_selected = []
        table_sel_json = ""
        col_sel_json = ""
        minimal_schema_text = ""

        retry_used = False
        sql_before_retry = ""
        sql_after_retry = ""
        retry_error_before = ""
        retry_error_after = ""

    llm_latency_ms = int((time.time() - t0) * 1000)

    # Execute expected + generated (generated already executed once in retry wrapper,
    # but we re-execute here to capture cols/rows for reporting consistently)
    exp_ok, exp_cols, exp_rows, exp_err = try_execute(expected_sql, fetch_rows=FETCH_ROWS)

    if generated_sql:
        gen_ok, gen_cols, gen_rows, gen_err = try_execute(generated_sql, fetch_rows=FETCH_ROWS)
    else:
        gen_ok, gen_cols, gen_rows, gen_err = False, None, None, "Empty SQL"

    expected_row_count = len(exp_rows) if exp_rows is not None else (0 if exp_ok else None)
    generated_row_count = len(gen_rows) if gen_rows is not None else (0 if gen_ok else None)

    exp_sample = rows_to_jsonable(exp_cols, exp_rows)
    gen_sample = rows_to_jsonable(gen_cols, gen_rows)

    exp_sample_json = dumps_safe(exp_sample) if exp_sample is not None else ""
    gen_sample_json = dumps_safe(gen_sample) if gen_sample is not None else ""

    exact_match = (expected_sql == generated_sql) if generated_sql else False

    result_match = False
    if exp_ok and gen_ok:
        result_match = rows_fingerprint(exp_cols, exp_rows) == rows_fingerprint(gen_cols, gen_rows)

    hint = result_diff_hint(
        exp_ok, exp_cols, exp_rows, exp_err,
        gen_ok, gen_cols, gen_rows, gen_err,
        exact_match, result_match
    )

    generated_ok = bool(gen_ok and (result_match or exact_match))

    results.append({
        "id": qid,
        "complexity": complexity,
        "question": question,

        "expected_sql": expected_sql,
        "generated_sql": generated_sql,

        "llm_latency_ms": llm_latency_ms,
        "llm_error": llm_err,

        "tables_selected": dumps_safe(tables_selected),
        "table_selector_json": table_sel_json,
        "column_selector_json": col_sel_json,
        "minimal_schema_text": minimal_schema_text,

        # Retry fields explained:
        "retry_used": retry_used,
        "sql_before_retry": sql_before_retry,
        "sql_after_retry": sql_after_retry,
        "retry_error_before": retry_error_before,
        "retry_error_after": retry_error_after,

        "expected_exec_ok": exp_ok,
        "expected_exec_error": exp_err or "",
        "generated_exec_ok": gen_ok,
        "generated_exec_error": gen_err or "",

        "expected_row_count_sample": expected_row_count,
        "generated_row_count_sample": generated_row_count,

        "expected_result_sample_json": exp_sample_json,
        "generated_result_sample_json": gen_sample_json,

        "exact_match": exact_match,
        "result_match_sample": result_match,
        "result_diff_hint": hint,

        "generated_ok": generated_ok
    })

df_report = pd.DataFrame(results)
df_report.head(3)

100%|███████████████████████████████████████████| 75/75 [19:07<00:00, 15.30s/it]


,id,complexity,question,expected_sql,generated_sql,llm_latency_ms,llm_error,tables_selected,table_selector_json,column_selector_json,...,generated_exec_ok,generated_exec_error,expected_row_count_sample,generated_row_count_sample,expected_result_sample_json,generated_result_sample_json,exact_match,result_match_sample,result_diff_hint,generated_ok
0,1,easy,List all products.,SELECT * FROM products;,SELECT * FROM public.products;,8152,,"[""public.products""]","{""tables"": [""public.products""], ""reason"": {""pu...","{""columns"": {""public.products"": [""product_id"",...",...,True,,20.0,20.0,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...",False,True,ok,True
1,2,easy,List all products in the 'Electronics' category.,SELECT * FROM public.products WHERE category =...,SELECT * FROM public.products WHERE category =...,9353,,"[""public.products""]","{""tables"": [""public.products""], ""reason"": {""pu...","{""columns"": {""public.products"": [""product_id"",...",...,True,,9.0,9.0,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...",True,True,ok,True
2,3,easy,Find all products priced above 100 dollars.,SELECT * FROM public.products WHERE price > 100;,SELECT * FROM public.products WHERE price > 100;,9115,,"[""public.products""]","{""tables"": [""public.products""], ""reason"": {""pu...","{""columns"": {""public.products"": [""product_id"",...",...,True,,8.0,8.0,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...","[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...",True,True,ok,True


In [25]:
OUT_REPORT = "nl2sql_test_report_with_results_EnrichKG_retry.csv"
OUT_SUMMARY = "nl2sql_test_report_summary_EnrichKG_retry.csv"

In [26]:
df_report.to_csv(OUT_REPORT, index=False)
print("Saved detailed report ✅:", OUT_REPORT)

Saved detailed report ✅: nl2sql_test_report_with_results_EnrichKG_retry.csv


In [27]:
summary = df_report.groupby("complexity").agg(
    total=("id", "count"),
    exact_acc=("exact_match","mean"),
    exec_ok_rate=("generated_exec_ok","mean"),
    result_acc=("result_match_sample","mean"),
    pass_rate=("generated_ok","mean"),
    avg_llm_ms=("llm_latency_ms","mean"),
).reset_index()

order = ["easy", "medium", "hard"]
summary["complexity"] = pd.Categorical(summary["complexity"], categories=order, ordered=True)
summary = summary.sort_values("complexity").reset_index(drop=True)

summary.to_csv(OUT_SUMMARY, index=False)
print("Saved summary ✅:", OUT_SUMMARY)

summary

Saved summary ✅: nl2sql_test_report_summary_EnrichKG_retry.csv


,complexity,total,exact_acc,exec_ok_rate,result_acc,pass_rate,avg_llm_ms
0,easy,25,0.16,1.00,0.80,0.80,10190.60
1,medium,25,0.00,0.84,0.28,0.28,12783.64
2,hard,25,0.00,0.76,0.12,0.12,22878.32


In [28]:
SQLCODER_SCHEMA = r"""
CREATE TABLE public.customers (
  customer_id UUID PRIMARY KEY, -- Unique ID for each customer
  first_name VARCHAR, -- Customer first name
  last_name VARCHAR, -- Customer last name
  email VARCHAR NOT NULL, -- Customer email address
  phone VARCHAR, -- Customer phone number
  address TEXT, -- Street address
  city VARCHAR, -- City
  state VARCHAR, -- State
  country VARCHAR, -- Country
  zip_code VARCHAR, -- Postal code
  created_at TIMESTAMP -- Created timestamp
);

CREATE TABLE public.orders (
  order_id UUID PRIMARY KEY, -- Unique ID for each order
  customer_id UUID NOT NULL, -- Customer who placed the order
  order_date TIMESTAMP, -- Order timestamp
  status VARCHAR NOT NULL, -- Order status (pending/processing/shipped/delivered)
  total_amount NUMERIC NOT NULL, -- Total order amount
  shipping_address TEXT, -- Shipping address
  notes TEXT -- Additional notes
);

CREATE TABLE public.order_items (
  order_item_id UUID PRIMARY KEY, -- Unique ID for each order item
  order_id UUID NOT NULL, -- Order this line belongs to
  product_id UUID NOT NULL, -- Product purchased
  quantity INTEGER NOT NULL, -- Units purchased
  unit_price NUMERIC NOT NULL, -- Unit price at purchase time
  subtotal NUMERIC NOT NULL, -- quantity * unit_price
  created_at TIMESTAMP -- Created timestamp
);

CREATE TABLE public.products (
  product_id UUID PRIMARY KEY, -- Unique ID for each product
  product_name VARCHAR NOT NULL, -- Name of the product
  description TEXT, -- Product description
  category VARCHAR NOT NULL, -- Product category
  price NUMERIC NOT NULL, -- Product price
  stock_quantity INTEGER NOT NULL, -- Inventory count
  created_at TIMESTAMP, -- Created timestamp
  updated_at TIMESTAMP -- Updated timestamp
);

CREATE TABLE public.reviews (
  review_id UUID PRIMARY KEY, -- Unique ID for each review
  product_id UUID NOT NULL, -- Reviewed product
  customer_id UUID NOT NULL, -- Customer who reviewed
  rating INTEGER NOT NULL, -- Rating score (1-5)
  review_text TEXT, -- Review text
  created_at TIMESTAMP -- Created timestamp
);

-- order_items.order_id can be joined with orders.order_id
-- order_items.product_id can be joined with products.product_id
-- orders.customer_id can be joined with customers.customer_id
-- reviews.product_id can be joined with products.product_id
-- reviews.customer_id can be joined with customers.customer_id
""".strip()


In [29]:
SQLCODER_INSTRUCTIONS = """### Instructions:
Your task is to convert a question into a SQL query, given a Postgres database schema.
Adhere to these rules:
- Use Table Aliases to prevent ambiguity in joins.
- Use only tables and columns that exist in the schema.
- Prefer explicit JOIN syntax over implicit joins.
- When creating a ratio, always cast the numerator as float.
- Return ONLY the SQL query. Do not include explanations, markdown, or code fences.
- Return exactly ONE SQL statement ending with a semicolon.
"""


In [30]:
def build_sqlcoder_prompt(question: str, schema_text: str = SQLCODER_SCHEMA) -> str:
    return f"""{SQLCODER_INSTRUCTIONS}

### Input:
Generate a SQL query that answers the question "{question}".
This query will run on a database whose schema is represented in this string:
{schema_text}

### Response:
"""

In [39]:
def normalize_sql(sql: str) -> str:
    if not sql:
        return ""

    sql = sql.strip()

    # Remove special tokens
    sql = re.sub(r"</?s>", "", sql)          # remove <s> and </s>
    sql = re.sub(r"<\|.*?\|>", "", sql)      # remove other special tokens if any

    # Remove markdown fences
    sql = re.sub(r"```sql", "", sql, flags=re.IGNORECASE)
    sql = re.sub(r"```", "", sql)

    sql = sql.strip()

    # Keep only first statement
    sql = sql.split(";")[0].strip() + ";"

    # Format consistently
    sql = sqlparse.format(
        sql,
        keyword_case="upper",
        strip_comments=True,
        reindent=False
    )

    sql = re.sub(r"\s+", " ", sql).strip()

    return sql

In [40]:
SQLCODER_MODEL = os.getenv("SQLCODER_MODEL", "sqlcoder:7b")

def sqlcoder_generate_sql(question: str, schema_text: str = SQLCODER_SCHEMA) -> str:
    prompt = build_sqlcoder_prompt(question, schema_text)
    raw = ollama.generate(
        model=SQLCODER_MODEL,
        prompt=prompt,
        options={"temperature": 0.0},
        stream=False
    ).get("response","").strip()
    return normalize_sql(raw)

In [47]:
results = []

for _, r in tqdm(df_tests.iterrows(), total=len(df_tests)):
    qid = int(r["id"])
    complexity = str(r["complexity"])
    question = str(r["question"])
    expected_sql = normalize_sql(str(r["expected_sql"]).strip())

    t0 = time.time()
    try:
        generated_sql = sqlcoder_generate_sql_with_kg(question)
        llm_error = ""
    except Exception as e:
        generated_sql = ""
        llm_error = str(e)
    llm_latency_ms = int((time.time() - t0) * 1000)

    exp_ok, exp_cols, exp_rows, exp_err = try_execute(expected_sql, fetch_rows=FETCH_ROWS)
    if generated_sql:
        gen_ok, gen_cols, gen_rows, gen_err = try_execute(generated_sql, fetch_rows=FETCH_ROWS)
    else:
        gen_ok, gen_cols, gen_rows, gen_err = False, None, None, "Empty SQL"

    expected_row_count = len(exp_rows) if exp_rows is not None else (0 if exp_ok else None)
    generated_row_count = len(gen_rows) if gen_rows is not None else (0 if gen_ok else None)

    exp_sample_json = dumps_safe(rows_to_jsonable(exp_cols, exp_rows)) if exp_ok else ""
    gen_sample_json = dumps_safe(rows_to_jsonable(gen_cols, gen_rows)) if gen_ok else ""

    exact_match = (expected_sql == generated_sql) if generated_sql else False
    result_match = False
    if exp_ok and gen_ok:
        result_match = rows_fingerprint(exp_cols, exp_rows) == rows_fingerprint(gen_cols, gen_rows)

    hint = result_diff_hint(
        exp_ok, exp_cols, exp_rows, exp_err,
        gen_ok, gen_cols, gen_rows, gen_err,
        exact_match, result_match
    )

    generated_ok = bool(gen_ok and (result_match or exact_match))

    results.append({
        "id": qid,
        "complexity": complexity,
        "question": question,
        "expected_sql": expected_sql,
        "generated_sql": generated_sql,
        "llm_latency_ms": llm_latency_ms,
        "llm_error": llm_error,
        "expected_exec_ok": exp_ok,
        "expected_exec_error": exp_err or "",
        "generated_exec_ok": gen_ok,
        "generated_exec_error": gen_err or "",
        "expected_row_count_sample": expected_row_count,
        "generated_row_count_sample": generated_row_count,
        "expected_result_sample_json": exp_sample_json,
        "generated_result_sample_json": gen_sample_json,
        "exact_match": exact_match,
        "result_match_sample": result_match,
        "result_diff_hint": hint,
        "generated_ok": generated_ok,
        "model": SQLCODER_MODEL
    })

df_sqlcoder_report = pd.DataFrame(results)
df_sqlcoder_report.to_csv("sqlcoder_report.csv", index=False)
df_sqlcoder_report.head(3)


100%|███████████████████████████████████████████| 75/75 [19:50<00:00, 15.87s/it]


,id,complexity,question,expected_sql,generated_sql,llm_latency_ms,llm_error,expected_exec_ok,expected_exec_error,generated_exec_ok,generated_exec_error,expected_row_count_sample,generated_row_count_sample,expected_result_sample_json,generated_result_sample_json,exact_match,result_match_sample,result_diff_hint,generated_ok,model
0,1,easy,List all products.,SELECT * FROM products;,,21394,,True,,False,Empty SQL,20.0,NaN,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...",,False,False,empty sql,False,sqlcoder:7b
1,2,easy,List all products in the 'Electronics' category.,SELECT * FROM public.products WHERE category =...,,11633,,True,,False,Empty SQL,9.0,NaN,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...",,False,False,empty sql,False,sqlcoder:7b
2,3,easy,Find all products priced above 100 dollars.,SELECT * FROM public.products WHERE price > 100;,,9585,,True,,False,Empty SQL,8.0,NaN,"[{""product_id"": ""a3bdf287-b2ee-4c58-b7e8-c7964...",,False,False,empty sql,False,sqlcoder:7b


In [48]:
OUT_REPORT = "nl2sql_test_report_with_results_sqlcoder_enrichKG.csv"
OUT_SUMMARY = "nl2sql_test_report_summary_sqlcoder_enrichKG.csv"

In [49]:
df_sqlcoder_report.to_csv(OUT_REPORT, index=False)
print("Saved detailed report ✅:", OUT_REPORT)

Saved detailed report ✅: nl2sql_test_report_with_results_sqlcoder_enrichKG.csv


In [44]:
summary = df_sqlcoder_report.groupby("complexity").agg(
    total=("id", "count"),
    exact_acc=("exact_match","mean"),
    exec_ok_rate=("generated_exec_ok","mean"),
    result_acc=("result_match_sample","mean"),
    pass_rate=("generated_ok","mean"),
    avg_llm_ms=("llm_latency_ms","mean"),
).reset_index()

order = ["easy", "medium", "hard"]
summary["complexity"] = pd.Categorical(summary["complexity"], categories=order, ordered=True)
summary = summary.sort_values("complexity").reset_index(drop=True)

summary.to_csv(OUT_SUMMARY, index=False)
print("Saved summary ✅:", OUT_SUMMARY)

summary

Saved summary ✅: nl2sql_test_report_summary_sqlcoder.csv


,complexity,total,exact_acc,exec_ok_rate,result_acc,pass_rate,avg_llm_ms
0,easy,25,0.00,1.00,0.08,0.08,5171.00
1,medium,25,0.04,0.92,0.24,0.24,6059.52
2,hard,25,0.00,0.88,0.08,0.08,6763.04


In [50]:
summary = df_sqlcoder_report.groupby("complexity").agg(
    total=("id", "count"),
    exact_acc=("exact_match","mean"),
    exec_ok_rate=("generated_exec_ok","mean"),
    result_acc=("result_match_sample","mean"),
    pass_rate=("generated_ok","mean"),
    avg_llm_ms=("llm_latency_ms","mean"),
).reset_index()

order = ["easy", "medium", "hard"]
summary["complexity"] = pd.Categorical(summary["complexity"], categories=order, ordered=True)
summary = summary.sort_values("complexity").reset_index(drop=True)

summary.to_csv(OUT_SUMMARY, index=False)
print("Saved summary ✅:", OUT_SUMMARY)

summary

Saved summary ✅: nl2sql_test_report_summary_sqlcoder_enrichKG.csv


,complexity,total,exact_acc,exec_ok_rate,result_acc,pass_rate,avg_llm_ms
0,easy,25,0.0,0.00,0.00,0.00,13024.16
1,medium,25,0.0,0.08,0.00,0.00,15220.28
2,hard,25,0.0,0.24,0.08,0.08,19282.28


In [45]:
def minimal_schema_to_create_tables(min_schema_text: str) -> str:
    # min_schema_text is like:
    # TABLE public.orders
    #  - order_id (uuid) PK ...
    # We convert into CREATE TABLE text.
    lines = min_schema_text.splitlines()
    out = []
    current_table = None
    cols = []
    for ln in lines:
        ln = ln.strip()
        if ln.startswith("TABLE "):
            if current_table and cols:
                out.append(f"CREATE TABLE {current_table} (\n  " + ",\n  ".join(cols) + "\n);\n")
            current_table = ln.replace("TABLE ","").strip()
            cols = []
        elif ln.startswith("- ") or ln.startswith("  - "):
            c = ln.split("--")[0].strip()
            c = c.replace("- ","").strip()
            # "col (type) PK NOT_NULL" -> "col type"
            m = re.match(r"([a-zA-Z0-9_]+)\s*\(([^)]+)\)", c)
            if m:
                col, typ = m.group(1), m.group(2)
                cols.append(f"{col} {typ}")
        elif ln.startswith("ALLOWED_JOINS"):
            break
    if current_table and cols:
        out.append(f"CREATE TABLE {current_table} (\n  " + ",\n  ".join(cols) + "\n);\n")
    # append join hints as comments
    out.append("\n-- JOINS\n")
    for ln in lines:
        if " = " in ln:
            out.append("-- " + ln.strip())
    return "\n".join(out).strip()

In [46]:
def sqlcoder_generate_sql_with_kg(question: str) -> str:
    pipe = nl2sql_pipeline(question)  # your KG selection pipeline
    schema_text = minimal_schema_to_create_tables(pipe["minimal_schema_text"])
    prompt = build_sqlcoder_prompt(question, schema_text=schema_text)
    raw = ollama.generate(model=SQLCODER_MODEL, prompt=prompt, options={"temperature": 0.0}).get("response","")
    return normalize_sql(raw)